In [1]:
import os, json, random, pathlib
import numpy as np
import tensorflow as tf

In [2]:
# ====== KONFIG ======
JSON_PATH   = "data.json"   # ganti sesuai
SEED        = 42
BATCH_SIZE  = 32
EPOCHS      = 80
VAL_SIZE    = 0.15
TEST_SIZE   = 0.15
USE_MIXED_PRECISION = True   # True jika GPU modern

In [3]:
# ====== SEED ======
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
if USE_MIXED_PRECISION:
    from tensorflow.keras import mixed_precision
    mixed_precision.set_global_policy("mixed_float16")

In [4]:
# ====== BACA JSON ======
with open(JSON_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

mfcc_list = data.get("mfcc")
labels    = data.get("labels")
mapping   = data.get("mapping", [])

if not isinstance(mfcc_list, list) or len(mfcc_list) == 0:
    raise ValueError("JSON tidak mengandung 'mfcc' yang valid.")
if labels is None:
    raise ValueError("JSON tidak memiliki 'labels'. Buat JSON lewat save_mfcc(...) yang menulis labels.")

In [5]:
# Bersihkan 'mapping' (Windows path jadi nama folder kelas saja)
def clean_name(p):
    return os.path.basename(p.replace("\\", "/"))
mapping = [clean_name(m) for m in mapping] if mapping else None

In [6]:
# ====== RAPIHKAN SHAPE & NORMALISASI ======
mfcc_arr = [np.array(x, dtype=np.float32) for x in mfcc_list]  # tiap item: (time, n_mfcc)
T = max(m.shape[0] for m in mfcc_arr)     # samakan panjang time
n_mfcc = mfcc_arr[0].shape[1]

def pad_or_crop(m, T=T):
    t = m.shape[0]
    if t == T: return m
    if t > T:
        s = (t - T)//2
        return m[s:s+T, :]
    out = np.zeros((T, m.shape[1]), dtype=np.float32)
    out[:t, :] = m
    return out

X = np.stack([pad_or_crop(m) for m in mfcc_arr], 0)     # (N, T, n_mfcc)
y = np.array(labels, dtype=np.int32)
num_classes = int(np.max(y)) + 1

In [7]:
# Standardisasi per-koefisien
mean = X.mean(axis=(0,1), keepdims=True)
std  = X.std(axis=(0,1), keepdims=True) + 1e-8
X    = (X - mean) / std

In [8]:
# Tambah channel untuk Conv2D
X = X[..., np.newaxis]                                   # (N, T, n_mfcc, 1)
input_shape = X.shape[1:]

In [9]:
# ====== SPLIT STRATIFIED ======
from sklearn.model_selection import train_test_split
Xtmp, Xtest, ytmp, ytest = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y)
val_ratio = VAL_SIZE / (1.0 - TEST_SIZE)
Xtrain, Xval, ytrain, yval = train_test_split(
    Xtmp, ytmp, test_size=val_ratio, random_state=SEED, stratify=ytmp)

print(f"[SPLIT] train={len(Xtrain)}, val={len(Xval)}, test={len(Xtest)}")

[SPLIT] train=415, val=89, test=90


In [16]:
# ====== tf.data ======
def make_ds(X, y, shuffle=False, batch=BATCH_SIZE):
    ds = tf.data.Dataset.from_tensor_slices((X.astype(np.float32), y.astype(np.int32)))
    if shuffle:
        ds = ds.shuffle(min(5000, len(X)), seed=SEED)
    return ds.batch(batch).prefetch(tf.data.AUTOTUNE)

train_ds = make_ds(Xtrain, ytrain, shuffle=True)
val_ds   = make_ds(Xval,   yval)
test_ds  = make_ds(Xtest,  ytest)

In [21]:
# ====== MODEL CNN (akurat & presisi) ======
from tensorflow.keras import layers, models, regularizers, metrics

def build_model(input_shape, n_classes):
    inp = layers.Input(shape=input_shape)
    x = inp
    for f in [32, 64, 128]:
        x = layers.Conv2D(f, (3,3), padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
        x = layers.Conv2D(f, (3,3), padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
        x = layers.MaxPool2D((2,2))(x)
        x = layers.Dropout(0.25)(x)
    x = layers.Conv2D(192, (3,3), padding="same", activation="relu")(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.35)(x)
    x = layers.Dense(256, activation="relu", kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.35)(x)
    out = layers.Dense(n_classes, activation="softmax", dtype="float32")(x)  # float32 for MP
    return models.Model(inp, out)

model = build_model(input_shape, num_classes)
opt = tf.keras.optimizers.Adam(learning_rate=3e-4)
model.compile(
    optimizer=opt,
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)           │ (None, 130, 13, 1)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_7 (Conv2D)                    │ (None, 130, 13, 32)         │             288 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_6                │ (None, 130, 13, 32)         │             128 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ re_lu_6 (ReLU)                       │ (None, 130, 13, 32)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_8 (Conv2D)                    │ (None, 130, 13, 32)         │           9,216 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_7                │ (None, 130, 13, 32)         │             128 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ re_lu_7 (ReLU)                       │ (None, 130, 13, 32)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_3 (MaxPooling2D)       │ (None, 65, 6, 32)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_5 (Dropout)                  │ (None, 65, 6, 32)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_9 (Conv2D)                    │ (None, 65, 6, 64)           │          18,432 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_8                │ (None, 65, 6, 64)           │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ re_lu_8 (ReLU)                       │ (None, 65, 6, 64)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_10 (Conv2D)                   │ (None, 65, 6, 64)           │          36,864 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_9                │ (None, 65, 6, 64)           │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ re_lu_9 (ReLU)                       │ (None, 65, 6, 64)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_4 (MaxPooling2D)       │ (None, 32, 3, 64)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_6 (Dropout)                  │ (None, 32, 3, 64)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_11 (Conv2D)                   │ (None, 32, 3, 128)          │          73,728 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_10               │ (None, 32, 3, 128)          │             512 │
│ (BatchNormalization)                 │                             │              

 Total params: 559,331 (2.13 MB)

 Trainable params: 558,435 (2.13 MB)

 Non-trainable params: 896 (3.50 KB)

In [22]:
# ====== CLASS WEIGHTS untuk presisi kelas minoritas ======
from sklearn.utils.class_weight import compute_class_weight
classes = np.unique(ytrain)
class_weights = compute_class_weight("balanced", classes=classes, y=ytrain)
class_weights = {int(c): float(w) for c, w in zip(classes, class_weights)}
print("[CLASS_WEIGHTS]", class_weights)

[CLASS_WEIGHTS] {0: 1.064102564102564, 1: 1.2575757575757576, 2: 0.7904761904761904}


In [23]:
# ====== CALLBACKS ======
pathlib.Path("checkpoints").mkdir(exist_ok=True)
callbacks = [
    tf.keras.callbacks.ModelCheckpoint("checkpoints/best.keras", monitor="val_accuracy", save_best_only=True, mode="max"),
    tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=10, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6),
]

In [24]:
# ====== TRAIN ======
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS,
                    class_weight=class_weights, callbacks=callbacks, verbose=1)

Epoch 1/80
13/13 ━━━━━━━━━━━━━━━━━━━━ 68s 5s/step - accuracy: 0.4241 - loss: 1.1000 - val_accuracy: 0.4719 - val_loss: 1.1066 - learning_rate: 3.0000e-04
Epoch 2/80
13/13 ━━━━━━━━━━━━━━━━━━━━ 69s 5s/step - accuracy: 0.5446 - loss: 0.9978 - val_accuracy: 0.4270 - val_loss: 1.0904 - learning_rate: 3.0000e-04
Epoch 3/80
13/13 ━━━━━━━━━━━━━━━━━━━━ 71s 5s/step - accuracy: 0.5325 - loss: 0.9902 - val_accuracy: 0.4270 - val_loss: 1.0928 - learning_rate: 3.0000e-04
Epoch 4/80
13/13 ━━━━━━━━━━━━━━━━━━━━ 70s 5s/step - accuracy: 0.5831 - loss: 0.9595 - val_accuracy: 0.4270 - val_loss: 1.1170 - learning_rate: 3.0000e-04
Epoch 5/80
13/13 ━━━━━━━━━━━━━━━━━━━━ 72s 6s/step - accuracy: 0.5446 - loss: 0.9534 - val_accuracy: 0.4270 - val_loss: 1.1570 - learning_rate: 3.0000e-04
Epoch 6/80
13/13 ━━━━━━━━━━━━━━━━━━━━ 71s 5s/step - accuracy: 0.6072 - loss: 0.8903 - val_accuracy: 0.4270 - val_loss: 1.2347 - learning_rate: 3.0000e-04
Epoch 7/80
13/13 ━━━━━━━━━━━━━━━━━━━━ 71s 5s/step - accuracy: 0.6048 - loss:

In [ ]:
# ====== EVALUASI ======
from sklearn.metrics import classification_report, confusion_matrix
yp = np.argmax(model.predict(test_ds), axis=1)
names = mapping if mapping and len(mapping)==num_classes else [str(i) for i in range(num_classes)]
print("\n=== TEST REPORT ===")
print(classification_report(ytest, yp, target_names=names, digits=4))
print("Confusion matrix:\n", confusion_matrix(ytest, yp))

In [ ]:
# ====== SIMPAN ======
model.save("model_mfcc_cnn.keras")
np.savez("mfcc_norm_stats.npz", mean=mean, std=std)
with open("label_map.json", "w", encoding="utf-8") as f:
    json.dump({"mapping": names, "num_classes": num_classes}, f, indent=2, ensure_ascii=False)
print("\n[OK] Saved: model_mfcc_cnn.keras, mfcc_norm_stats.npz, label_map.json")